In [2]:
!pip install -q "datasets<3" transformers seqeval

In [3]:
import numpy as np
from datasets import load_dataset

raw = load_dataset("nlpaueb/finer-139")
label_names = raw["train"].features["ner_tags"].feature.names

def split_stats(ds):
    n = len(ds)
    o_only = sum(1 for tags in ds["ner_tags"] if not any(t != 0 for t in tags))
    return n, o_only

before = {s: split_stats(raw[s]) for s in ["train","validation","test"]}
print("BEFORE — total sentences | O-only sentences")
for s in ["train","validation","test"]:
    print(f"  {s}: {before[s][0]:,} | {before[s][1]:,}")

Generating train split:   0%|          | 0/900384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/112494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/108378 [00:00<?, ? examples/s]

BEFORE — total sentences | O-only sentences
  train: 900,384 | 721,189
  validation: 112,494 | 90,891
  test: 108,378 | 89,589


In [4]:
import random
random.seed(42)
SEP = "\u241F"
def sigs(ds): return [SEP.join(t) for t in ds["tokens"]]

clean = {}

# 1) Tekrarları sil (her split kendi içinde)
def dedup(ds):
    seen=set(); keep=[]
    for i,s in enumerate(sigs(ds)):
        if s not in seen:
            seen.add(s); keep.append(i)
    return ds.select(keep)
for s in ["train","validation","test"]:
    clean[s] = dedup(raw[s])
dedup_removed = {s: len(raw[s])-len(clean[s]) for s in clean}

# 2) Sızıntıyı gider: val/test'te de bulunan train cümlelerini train'den at
vt = set(sigs(clean["validation"])) | set(sigs(clean["test"]))
ts = sigs(clean["train"])
keep = [i for i,sg in enumerate(ts) if sg not in vt]
leak_removed = len(ts) - len(keep)
clean["train"] = clean["train"].select(keep)

# 3) O-only azalt — SADECE train. Etiketli cümleler hep kalır, O-only'nin %30'u kalır
KEEP = 0.30
idx=[]
for i,tags in enumerate(clean["train"]["ner_tags"]):
    if any(t!=0 for t in tags) or random.random()<KEEP:
        idx.append(i)
oonly_removed = len(clean["train"]) - len(idx)
clean["train"] = clean["train"].select(idx)

print("Deduplication removed:", dedup_removed)
print("Leakage sentences removed from train:", f"{leak_removed:,}")
print("O-only sentences dropped from train:", f"{oonly_removed:,}")
print("Cleaning done.")

Deduplication removed: {'train': 296717, 'validation': 11369, 'test': 13348}
Leakage sentences removed from train: 41,099
O-only sentences dropped from train: 300,958
Cleaning done.


In [5]:
after = {s: split_stats(clean[s]) for s in ["train","validation","test"]}

print("=== CLEANING SUMMARY: sentence counts ===")
print(f"{'Split':<12}{'Before':>12}{'After':>12}{'Removed':>12}{'% kept':>9}")
for s in ["train","validation","test"]:
    b,a = before[s][0], after[s][0]
    print(f"{s:<12}{b:>12,}{a:>12,}{b-a:>12,}{a/b*100:>8.1f}%")

print("\n=== O-only (no-entity) sentences ===")
print(f"{'Split':<12}{'Before':>12}{'After':>12}")
for s in ["train","validation","test"]:
    print(f"{s:<12}{before[s][1]:>12,}{after[s][1]:>12,}")

# Removal breakdown (train)
print("\n=== Train removal breakdown ===")
print(f"  Duplicates removed : {dedup_removed['train']:,}")
print(f"  Leakage removed    : {leak_removed:,}")
print(f"  O-only dropped     : {oonly_removed:,}")

=== CLEANING SUMMARY: sentence counts ===
Split             Before       After     Removed   % kept
train            900,384     261,610     638,774    29.1%
validation       112,494     101,125      11,369    89.9%
test             108,378      95,030      13,348    87.7%

=== O-only (no-entity) sentences ===
Split             Before       After
train            721,189     128,887
validation        90,891      81,043
test              89,589      77,464

=== Train removal breakdown ===
  Duplicates removed : 296,717
  Leakage removed    : 41,099
  O-only dropped     : 300,958


In [6]:
import os
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def align_labels(example):
    tok = tokenizer(example["tokens"], truncation=True, max_length=128, is_split_into_words=True)
    word_ids = tok.word_ids(); labels=[]; prev=None
    for wid in word_ids:
        if wid is None: labels.append(-100)
        elif wid != prev: labels.append(example["ner_tags"][wid])
        else: labels.append(-100)
        prev = wid
    tok["labels"] = labels
    return tok

tokenized = {s: clean[s].map(align_labels, remove_columns=clean[s].column_names,
                             desc=f"Tokenizing {s}") for s in ["train","validation","test"]}

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
for s in ["train","validation","test"]:
    raw[s].to_parquet(f"data/raw/{s}.parquet")        # ham (değişmedi)
    clean[s].to_parquet(f"data/cleaned/{s}.parquet")  # temizlenmiş (dedup+sızıntı+O azaltma)
    tokenized[s].to_parquet(f"data/processed/{s}.parquet")  # temiz halin tokenize edilmişi
print("Saved: data/raw, data/cleaned, data/processed")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Tokenizing train:   0%|          | 0/261610 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/102 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/109 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/96 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/96 [00:00<?, ?ba/s]

Saved: data/raw, data/cleaned, data/processed


In [7]:
readme = (
"# FiNER-139: Financial Numeric Entity Recognition (XBRL Tagging)\n\n"
"Token classification on SEC financial filings. ~95% of labeled tokens are numbers, "
"so the correct XBRL tag depends on context, not the number itself.\n\n"
"## Dataset\n"
"- Source: nlpaueb/finer-139 (ACL 2022) — https://huggingface.co/datasets/nlpaueb/finer-139\n"
"- Original: 1.1M sentences (900,384 / 112,494 / 108,378)\n"
"- 139 entity types -> 279 BIO tags; 167 active in train\n"
"- data/raw/      = original, untouched\n"
"- data/cleaned/  = after cleaning (dedup + leakage removal + O-only reduction)\n"
"- data/processed/= cleaned data tokenized (bert-base-cased) + label-aligned, model-ready\n\n"
"## Cleaning (train / val / test)\n"
"Aggressive cleaning only on TRAIN; val/test kept close to original for honest evaluation.\n\n"
"| Split | Before | After | % kept |\n"
"|-------|--------|-------|--------|\n"
"| train | 900,384 | 261,610 | 29.1% |\n"
"| validation | 112,494 | 101,125 | 89.9% |\n"
"| test | 108,378 | 95,030 | 87.7% |\n\n"
"Train removal breakdown: 296,717 duplicates + 41,099 train/val-test leakage + 300,958 O-only sentences.\n"
"O-only (no-entity) train sentences: 721,189 -> 128,887 (80% -> 49%).\n\n"
"## Key facts\n"
"- Severe imbalance: 99.22% of tokens are O\n"
"- 94.7% of labeled tokens numeric\n"
"- 10.6% of words split into sub-pieces (number fragmentation)\n"
"- max_length = 128 (covers 97.4% of sentences)\n\n"
"## Evaluation\n"
"Entity-level F1 (seqeval). Accuracy is misleading due to the O majority.\n\n"
"## Team\nSee CONTRIBUTORS.md.\n")
open("README.md","w").write(readme)

import zipfile, os
files = ["README.md"]
for sub in ["raw","cleaned","processed"]:
    for s in ["train","validation","test"]:
        files.append(f"data/{sub}/{s}.parquet")
with zipfile.ZipFile("finer139_repo.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f): z.write(f)
print("ZIP ready:", round(os.path.getsize("finer139_repo.zip")/1024/1024,1), "MB")
print("Files:", len(files))

ZIP ready: 174.4 MB
Files: 10


In [8]:
import os

# Üç klasörde ne var, gör
for sub in ["raw","cleaned","processed"]:
    p = f"data/{sub}"
    if os.path.exists(p):
        print(f"{sub}: {sorted(os.listdir(p))}")
    else:
        print(f"{sub}: YOK")

raw: ['test.parquet', 'train.parquet', 'validation.parquet']
cleaned: ['test.parquet', 'train.parquet', 'validation.parquet']
processed: ['test.parquet', 'train.parquet', 'validation.parquet']


In [9]:
import zipfile, os

# Eski zip'i sil
if os.path.exists("finer139_repo.zip"):
    os.remove("finer139_repo.zip")

files = ["README.md"]
for sub in ["raw","cleaned","processed"]:
    for s in ["train","validation","test"]:
        files.append(f"data/{sub}/{s}.parquet")

with zipfile.ZipFile("finer139_repo.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)

# Doğrula: zip'in içinde gerçekten cleaned var mı?
with zipfile.ZipFile("finer139_repo.zip") as z:
    names = z.namelist()
print("ZIP içeriği:")
for n in names: print("  ", n)
print("\nBoyut:", round(os.path.getsize("finer139_repo.zip")/1024/1024,1), "MB")

ZIP içeriği:
   README.md
   data/raw/train.parquet
   data/raw/validation.parquet
   data/raw/test.parquet
   data/cleaned/train.parquet
   data/cleaned/validation.parquet
   data/cleaned/test.parquet
   data/processed/train.parquet
   data/processed/validation.parquet
   data/processed/test.parquet

Boyut: 174.4 MB
